# 02 — Image Generation (SD 1.5 + ControlNet MLSD)

> **Requires T4 or better GPU runtime.**
> Go to **Runtime → Change runtime type → T4 GPU** before running any cell.

This notebook runs the full generation experiment:

| Step | What happens |
|------|-------------|
| Cells 1–3 | Confirm GPU, mount Drive, install deps |
| Cell 4 | Log in to HuggingFace (needed for `runwayml/stable-diffusion-v1-5`) |
| Cell 5–6 | Load config + initialise SQLite DB |
| Cell 7 | **Pilot**: one image end-to-end to check VRAM and timing |
| Cell 8 | **Full factorial**: `(scene × strategy × control_mode × seed)` |
| Cell 9 | Back up outputs to Drive |

Run cells in order.  Do **not** skip the pilot (Cell 7) — it surfaces OOM before wasting quota on a full run.

---

## Cell 1 — Confirm GPU

In [ ]:
import torch

# Print driver / GPU summary
!nvidia-smi

# Sanity-check from Python side
is_available = torch.cuda.is_available()
print(f"\ntorch.cuda.is_available() = {is_available}")

if not is_available:
    raise RuntimeError(
        "No CUDA GPU detected.\n"
        "Go to Runtime → Change runtime type → T4 GPU, then reconnect."
    )

device_name = torch.cuda.get_device_name(0)
total_vram  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Device : {device_name}")
print(f"VRAM   : {total_vram:.1f} GB")

if total_vram < 12:
    print("⚠  Less than 12 GB VRAM detected.  Generation may OOM at 512×512 fp16.")
else:
    print("✓  VRAM looks sufficient for SD 1.5 + ControlNet at 512×512 fp16.")

## Cell 2 — Mount Drive, Clone / Pull Repo, Install Dependencies

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/ikea-sd")
REPO_DIR   = Path("/content/ikea-sd")

# ── EDIT THIS LINE: replace with your actual GitHub repo URL ──────────────
REPO_URL = "https://github.com/YOUR_USERNAME/ikea-sd.git"
# ─────────────────────────────────────────────────────────────────────────

if REPO_DIR.is_dir():
    print("Repo already cloned — pulling latest changes.")
    os.system(f"git -C {REPO_DIR} pull --ff-only")
else:
    ret = os.system(f"git clone {REPO_URL} {REPO_DIR}")
    if ret != 0:
        raise RuntimeError(
            f"git clone failed (exit {ret}). Check that REPO_URL is set."
        )

# Change to repo root so relative paths in config.yaml resolve correctly
%cd /content/ikea-sd

# Install dependencies — skip torch/torchvision (pre-installed on Colab T4)
with open("requirements.txt") as f:
    reqs = [
        ln.strip()
        for ln in f
        if ln.strip() and not ln.startswith("#") and not ln.lower().startswith("torch")
        and not ln.lower().startswith("numpy")
        and not ln.lower().startswith("pandas")
    ]

print(f"Installing {len(reqs)} packages (torch/torchvision excluded)…")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *reqs],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    raise RuntimeError("pip install failed.")

print("Dependencies ready.")

## Cell 3 — HuggingFace Login

`runwayml/stable-diffusion-v1-5` requires accepting a licence on HuggingFace.
Store your token as a Colab secret named `HF_TOKEN` (**Secrets** panel, left sidebar) and it will be read without appearing in the notebook output.

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami

# Read token from Colab Secrets (never paste tokens directly into notebooks)
try:
    hf_token = userdata.get("HF_TOKEN")
except userdata.SecretNotFoundError:
    raise RuntimeError(
        "HF_TOKEN secret not found.\n"
        "1. Click the key icon in the left sidebar.\n"
        "2. Add a secret named HF_TOKEN with your token from "
        "https://huggingface.co/settings/tokens\n"
        "3. Re-run this cell."
    )

login(token=hf_token, add_to_git_credential=False)

# Confirm identity (token is never printed)
identity = whoami()
print(f"Logged in as: {identity['name']}")

## Cell 4 — Load Config

In [ ]:
import sys
from pathlib import Path
import yaml

REPO_DIR = Path("/content/ikea-sd")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

with open(REPO_DIR / "config.yaml") as f:
    cfg = yaml.safe_load(f)

# Override paths to use Colab-local outputs/ and Drive-backed DB + data
cfg["paths"]["data_raw"]       = str(REPO_DIR / "data" / "raw")
cfg["paths"]["data_processed"] = str(REPO_DIR / "data" / "processed")
cfg["paths"]["outputs"]        = str(REPO_DIR / "outputs" / "generated_images")
# Keep the DB on Drive so it survives session restarts
cfg["paths"]["db_path"]        = "/content/drive/MyDrive/ikea-sd/results.db"

print("Config loaded:")
for section, values in cfg.items():
    print(f"  [{section}]")
    if isinstance(values, dict):
        for k, v in values.items():
            print(f"      {k}: {v}")
    else:
        print(f"      {values}")

## Cell 5 — Initialise Database

In [ ]:
from pathlib import Path
from src.db import init_db, query_df

db_path = cfg["paths"]["db_path"]
conn    = init_db(db_path)

# Report current DB state (useful on resume: shows how many runs already exist)
runs_df = query_df(conn, "SELECT COUNT(*) AS total_runs FROM runs")
print(f"Database : {db_path}")
print(f"Runs already recorded : {runs_df.iloc[0]['total_runs']}")

conn.close()
print("DB ready.")

## Cell 6 — Pilot Run (1 image, strategy A, uncontrolled)

Generates a single image to verify the pipeline loads correctly, measure
wall-clock time per image, and confirm VRAM stays within budget.
**Do not skip this cell.** A failure here is cheap; a failure in Cell 7 after
100 images is not.

In [ ]:
import gc
import time
from pathlib import Path

import torch
import matplotlib.pyplot as plt

from src.generator import Generator
from src.utils import timer_context

PILOT_OUT = Path(cfg["paths"]["outputs"]) / "pilot"
PILOT_OUT.mkdir(parents=True, exist_ok=True)

# ── VRAM snapshot before loading ──────────────────────────────────────────
def _vram_used_gb() -> float:
    if torch.cuda.is_available():
        return torch.cuda.memory_reserved(0) / 1024**3
    return 0.0

vram_before = _vram_used_gb()
print(f"VRAM before model load : {vram_before:.2f} GB")

gen = Generator(cfg)

# ── Pilot: one uncontrolled image ─────────────────────────────────────────
PILOT_PROMPT = "a photo of a scandinavian living room, containing sofa and bookshelf, realistic interior"

t0 = time.perf_counter()
with timer_context("Pilot generation"):
    image = gen.generate_uncontrolled(
        positive=PILOT_PROMPT,
        negative=None,
        seed=42,
    )
elapsed = time.perf_counter() - t0

vram_after = _vram_used_gb()

# ── Save and display ──────────────────────────────────────────────────────
pilot_path = PILOT_OUT / "pilot_A_none_seed42.png"
image.save(pilot_path)

print(f"\nPilot image   : {pilot_path}")
print(f"Time          : {elapsed:.1f} s  (~{elapsed:.0f}s per image)")
print(f"VRAM before   : {vram_before:.2f} GB")
print(f"VRAM after    : {vram_after:.2f} GB  (reserved by PyTorch allocator)")

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(image)
ax.set_title(f"Pilot — Strategy A, no control\n{PILOT_PROMPT[:60]}…", fontsize=8)
ax.axis("off")
plt.tight_layout()
plt.show()

# ── Free gen instance between pilot and full run ──────────────────────────
gen.unload()
del gen
gc.collect()
torch.cuda.empty_cache()
print(f"\nVRAM after unload : {_vram_used_gb():.2f} GB")
print("Pilot complete — safe to proceed to Cell 7.")

## Cell 7 — Full Factorial Run

Runs all `(scene × strategy × control_mode × seed)` combinations.

**Start small** (`NUM_SCENES = 5`) to verify end-to-end correctness before
committing to a longer run.  The tqdm bar shows images/s and an ETA.
Completed images are skipped automatically on re-run (resume-safe).

In [ ]:
import json
import logging
from pathlib import Path

from src.generator import run_experiment

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s  %(name)s  %(message)s",
    force=True,
)

# ── Experiment parameters — edit these before a longer run ────────────────
NUM_SCENES    = 5          # increase to 15 or 30 for the final report
SEEDS         = [42, 123, 7]
STRATEGIES    = ["A", "B", "C", "D"]
CONTROL_MODES = ["none", "mlsd"]
# ─────────────────────────────────────────────────────────────────────────

# Load scene records (produced by notebook 01)
processed_json = Path(cfg["paths"]["data_processed"]) / "structured_scenes.json"
if not processed_json.is_file():
    # Fall back to Drive copy
    drive_json = Path("/content/drive/MyDrive/ikea-sd/data/processed/structured_scenes.json")
    if drive_json.is_file():
        processed_json = drive_json
    else:
        raise FileNotFoundError(
            f"structured_scenes.json not found at {processed_json} or on Drive.\n"
            "Run notebook 01 first."
        )

with open(processed_json, encoding="utf-8") as f:
    all_scenes = json.load(f)

scenes = all_scenes[:NUM_SCENES]
print(f"Scenes selected : {len(scenes)} / {len(all_scenes)}")
print(f"Seeds           : {SEEDS}")
print(f"Strategies      : {STRATEGIES}")
print(f"Control modes   : {CONTROL_MODES}")
total = len(scenes) * len(SEEDS) * len(STRATEGIES) * len(CONTROL_MODES)
print(f"Total images    : {total}")
print()

run_experiment(
    scenes=scenes,
    seeds=SEEDS,
    db_path=cfg["paths"]["db_path"],
    out_dir=cfg["paths"]["outputs"],
    config=cfg,
    strategies=STRATEGIES,
    control_modes=CONTROL_MODES,
)

print(f"\nRun complete.  Images written to: {cfg['paths']['outputs']}")

## Cell 8 — Back Up Outputs to Drive

In [ ]:
from pathlib import Path

LOCAL_OUTPUTS = Path("/content/ikea-sd/outputs")
DRIVE_OUTPUTS = Path("/content/drive/MyDrive/ikea-sd/outputs")
DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)

print(f"Syncing {LOCAL_OUTPUTS}  →  {DRIVE_OUTPUTS}")
!rsync -av --progress /content/ikea-sd/outputs/ /content/drive/MyDrive/ikea-sd/outputs/

# Count synced images
n_local = len(list(LOCAL_OUTPUTS.rglob("*.png")))
n_drive = len(list(DRIVE_OUTPUTS.rglob("*.png")))
print(f"\nLocal  PNGs : {n_local}")
print(f"Drive  PNGs : {n_drive}")
print("Back-up complete.")

## Checkpoint

Cell 8 (`rsync`) just confirmed all generated images are on Drive.

**If this worked with `NUM_SCENES = 5`**, re-open Cell 7, change:
```python
NUM_SCENES = 15   # or 30 for the final report
```
and re-run.  Already-completed images will be skipped automatically.

**Estimated time on T4** (25 inference steps, 512×512, fp16):
- Uncontrolled image: ~8–12 s
- Controlled (MLSD): ~10–15 s (MLSD preprocessing adds ~1 s)
- 30 scenes × 4 strategies × 2 modes × 3 seeds = 720 images ≈ 2–3 hours

---

## 🛟 Recovery

### If Colab disconnects mid-run

Your recovery point is **Drive**.  Both the SQLite database (`results.db`) and
the generated images (`outputs/`) are written directly to Drive during the run.

On reconnect:
1. Re-run **all cells in order** (Cells 1–6 are fast and idempotent).
2. Re-run **Cell 7** — the generator checks whether each output PNG already
   exists on disk **before** calling the diffusion pipeline:
   ```python
   if img_path.is_file():
       pbar.update(1)
       continue          # skip, don't regenerate
   ```
   The `image_path` column also has a `UNIQUE` constraint in the DB, so
   even if a PNG somehow exists without a DB row (or vice versa), a duplicate
   insert will raise `IntegrityError` and be caught gracefully.
3. The tqdm bar will fast-forward through the skipped images and resume from
   where it left off.

### If you hit OOM mid-run

The generator catches `RuntimeError("out of memory …")` per image, flushes
the cache, and continues.  If the session itself crashes, free VRAM manually
before restarting the run:

```python
import gc, torch

gc.collect()
torch.cuda.empty_cache()

# Optional: report how much is now free
free  = torch.cuda.mem_get_info()[0] / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"VRAM free: {free:.1f} / {total:.1f} GB")
```

Then re-run Cell 7 — it will skip completed images and resume from the first
one that was not yet saved.